# T02 · Géométrie analytique — ADIF72

**Objectifs :**
- Normes (L1, L2, L∞, Minkowski) et distances associées
- Produit scalaire, orthogonalité, angles
- Projections et bases orthonormales (Gram-Schmidt)
- Similarité cosinus (applications : recherche, embeddings)

**Pourquoi c'est utile :**
| Besoin data science | Outil géométrique |
|---|---|
| Distance entre points (KNN, clustering) | L2, Manhattan, L∞ |
| Similarité (recherche, RAG, embeddings) | Cosinus, corrélation |
| Orthogonalisation (PCA, QR, SVD) | Gram-Schmidt |
| Reg (Lasso, ElasticNet) | Norme L1 (sparse) |


In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=100)
print("Prêt pour la géométrie. numpy", np.__version__)

Prêt pour la géométrie. numpy 2.5.4


## 1 · Normes d'un vecteur

Une **norme** est une fonction `‖·‖` qui mesure la « longueur » d'un vecteur :
1. `‖x‖ ≥ 0`, et `‖x‖ = 0 ⇔ x = 0`
2. `‖αx‖ = |α| ‖x‖` (homogénéité)
3. `‖x+y‖ ≤ ‖x‖ + ‖y‖` (inégalité triangulaire)

Les 3 normes usuelles :
| Norme | Formule | Usage typique |
|---|---|---|
| **L1** (Manhattan) | `Σ\|xᵢ\|` | Lasso (sparcity) |
| **L2** (Euclidienne) | `√(Σ xᵢ²)` | Moindres carrés |
| **L∞** (max) | `max\|xᵢ\|` | Bornes sup |

La **Minkowski** `‖x‖_p = (Σ \|xᵢ\|^p)^{1/p}` généralise L1 (p=1), L2 (p=2), L∞ (p→∞).

In [2]:
x = np.array([3.0, -4.0, 2.0])
print("x =", x)
print("L1      :", np.linalg.norm(x, ord=1))
print("L2      :", np.linalg.norm(x, ord=2))
print("L∞      :", np.linalg.norm(x, ord=np.inf))
print("Mink p=4:", np.linalg.norm(x, ord=4))

x = [ 3. -4.  2.]
L1      : 9.0
L2      : 5.385164807134504
L∞      : 4.0
Mink p=4: 4.334546600055874


### Exercice corrigé

In [3]:
# ---- Exercice P1 : normes de a=(5,-1,2) ----
a1 = np.array([5.0, -1.0, 2.0])
print(f"P1-ex : L1={np.linalg.norm(a1,1)}, L2={np.linalg.norm(a1,2):.3f}, Linf={np.linalg.norm(a1,np.inf)}")
assert np.linalg.norm(a1, 1) == 8


P1-ex : L1=8.0, L2=5.477, Linf=5.0


## 2 · Produit scalaire, cosinus, angle

Le **produit scalaire** (dot product) de `u, v ∈ ℝⁿ` :
```
u·v = Σ uᵢ vᵢ  =  ‖u‖‖v‖ cos(θ)
```
C'est l'opération la plus importante en géométrie :
- `u·v = 0`  ⇔  **orthogonalité**
- `cos θ > 0`  ⇒  angle aigu (< 90°)
- `cos θ = 0`  ⇒  perpendiculaires
- `cos θ < 0`  ⇒  angle obtus (> 90°)

Le **produit vectoriel** en 3D donne un vecteur perpendiculaire à `u` et `v`
de norme `‖u‖‖v‖ sin θ` (direction = règle de la main droite).

In [4]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, -1.0, 2.0])
dot = u @ v
cos = dot / (np.linalg.norm(u) * np.linalg.norm(v))
angle = np.degrees(np.arccos(np.clip(cos, -1, 1)))
print(f"u·v = {dot}")
print(f"cos(θ) = {cos:.4f}")
print(f"θ = {angle:.2f}°")
# produit vectoriel
print("u × v =", np.cross(u, v))
assert np.isclose((np.cross(u, v)) @ u, 0)  # perpendiculaire à u
assert np.isclose((np.cross(u, v)) @ v, 0)  # et à v

u·v = 8.0
cos(θ) = 0.4666
θ = 62.19°
u × v = [ 7. 10. -9.]


### Exercice corrigé

In [5]:
# ---- Exercice P2 : angle entre p=(1,0,0) et q=(0,1,0) ----
p = np.array([1.0, 0, 0]); q = np.array([0.0, 1, 0])
ang = np.degrees(np.arccos(p @ q / (np.linalg.norm(p) * np.linalg.norm(q))))
print(f"P2-ex : angle(p,q) = {ang} deg (orthogonaux)")
assert ang == 90


P2-ex : angle(p,q) = 90.0 deg (orthogonaux)


## 3 · Orthogonalité & projection

**Projection orthogonale** de `u` sur `v` :
```
proj_v(u) = (u·v)/(v·v) · v
```
Le résidu `u - proj_v(u)` est **perpendiculaire** à `v`.
C'est la base de la géométrie des moindres carrés : on projette `y`
sur l'image de `X` pour trouver le `θ̂` qui minimise `‖Xθ - y‖`.

**Matrice de projection** sur un sous-espace `W` (spannés par les colonnes
d'une matrice `Q` de colonnes orthogonales) :
```
P_W = Q (Qᵀ Q)⁻¹ Qᵀ
```
Propriétés : `P² = P` (idempotente), `Pᵀ = P` (symétrique), `spec(P) ⊂ {0,1}`.

In [6]:
def proj(u, v):
    return (u @ v) / (v @ v) * v

u2 = np.array([2.0, 1.0]); v2 = np.array([1.0, 0.0])
p = proj(u2, v2)
print("proj_v(u) =", p)
print("Résidu = u - proj =", u2 - p)
assert np.isclose((u2 - p) @ v2, 0)   # perpendiculaire

# Matrice de projection sur le plan span(e1, e2) dans R3
Q = np.array([[1, 0], [0, 1], [0, 0]])   # plan xy
P = Q @ np.linalg.inv(Q.T @ Q) @ Q.T
print("\nProjection sur le plan xy (R^3 -> R^3) :\n", P)
v3 = np.array([1.0, 2.0, 5.0])
print("P @ [1,2,5] =", P @ v3, " (composante z annulée)")
assert np.allclose(P @ P, P)  # idempotente
assert np.allclose(P, P.T)    # symétrique

proj_v(u) = [2. 0.]
Résidu = u - proj = [0. 1.]

Projection sur le plan xy (R^3 -> R^3) :
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]
P @ [1,2,5] = [1. 2. 0.]  (composante z annulée)


### Exercice corrigé

In [7]:
# ---- Exercice P3 : projection de m=(3,4) sur n=(1,2) ----
m = np.array([3.0, 4.0]); n = np.array([1.0, 2.0])
coef = (m @ n) / (n @ n); pr = coef * n
print(f"P3-ex : coef = {coef}, proj = {pr} ; residu orthogonal ? {np.isclose((m-pr)@n, 0)}")
assert np.allclose(pr, [2.2, 4.4])


P3-ex : coef = 2.2, proj = [2.2 4.4] ; residu orthogonal ? True


## 4 · Distances entre points

| Distance | Formule | Sens |
|---|---|---|
| **Euclidienne** | `‖a - b‖₂` | « ligne droite » (physique, géométrie) |
| **Manhattan / L1** | `Σ \|aᵢ-bᵢ\|` | « rue de la ville » (grille) |
| **Chebyshev / L∞** | `max \|aᵢ-bᵢ\|` | « roque d'échec » (mouvement 3D) |
| **Corrélation** | `1 - r(a,b)` | Indépendante à dérive/échelle (séries) |
| **Cosinus** | `1 - cos` | Indépendante à l'échelle (textes, embeddings) |

En clustering / KNN, le choix de la distance **change le résultat**.
La corrélation et le cosinus sont les plus robustes en data science.

In [8]:
a = np.array([1.0, 2.0, 3.0]); b = np.array([4.0, 6.0, 3.0])
print(f"Euclidienne  = {np.linalg.norm(a - b):.3f}")
print(f"Manhattan    = {np.abs(a - b).sum():.3f}")
print(f"Chebyshev    = {np.max(np.abs(a - b)):.3f}")

# Distance de corrélation (1 - cosinus des vectres centrés)
def dist_correlation(x, y):
    return 1 - np.corrcoef(x, y)[0, 1]

xs = np.array([1.0, 2, 3, 4, 5])
ys = np.array([1.1, 2.2, 2.9, 4.1, 4.8])
print(f"Corrélation  = {dist_correlation(xs, ys):.5f} (0 = parfaitement corrélées)")

Euclidienne  = 5.000
Manhattan    = 7.000
Chebyshev    = 4.000
Corrélation  = 0.00339 (0 = parfaitement corrélées)


### Exercice corrigé

In [9]:
# ---- Exercice P4 : distance + cosinus + projection combines ----
a = np.array([1.0, 2, 3]); b = np.array([4.0, 5, 6]); c = np.array([7.0, 8, 10])
print(f"P4-ex : dist(a,b) = {np.linalg.norm(a-b):.4f} (= 3*sqrt(3) = {3*np.sqrt(3):.4f})")
assert np.isclose(np.linalg.norm(a-b), 3*np.sqrt(3))
cos_ab = a @ b / (np.linalg.norm(a)*np.linalg.norm(b))
print(f"       cos(a,b) = {cos_ab:.4f} (= 32/sqrt(1078))")
assert np.isclose(cos_ab, 32/np.sqrt(1078))
proj_c = (c @ a) / (a @ a) * a
print(f"       proj_a(c) = {np.round(proj_c, 4)}  (coef = 53/14 = {53/14:.4f})")
assert np.allclose(proj_c, (53/14)*np.array([1.0, 2, 3]))


P4-ex : dist(a,b) = 5.1962 (= 3*sqrt(3) = 5.1962)
       cos(a,b) = 0.9746 (= 32/sqrt(1078))
       proj_a(c) = [ 3.786  7.571 11.357]  (coef = 53/14 = 3.7857)


## 5 · Bases orthonormales — Gram-Schmidt

Une **base orthonormale** : des vecteurs de norme 1 mutuellement
perpendiculaires. C'est la « meilleure » base pour :
- diagonaliser une matrice symétrique (théorème spectral)
- décomposer une matrice en QR (`Q` orthogonale, `R` triangulaire)
- le SVD (`U` et `V` sont orthonormales)

**Algorithme de Gram-Schmidt** : orthonormalise une base quelconque
en soustrayant, un à un, la composante projetée sur les précédents.

Variantes :
- **Classique** (ce que nous implémentons) — instable numériquement si les
  vecteurs sont presque parallèles.
- **Modifiée** (MGS) — plus stable, c'est celle qu'utilise `scipy.linalg.qr`.

In [10]:
def gram_schmidt(V):
    """Orthonormalise les colonnes de V."""
    Q = []
    for v in V.T:
        q = v.copy()
        for w in Q:
            q = q - (q @ w) * w
        n = np.linalg.norm(q)
        if n < 1e-12:
            continue
        q = q / n
        Q.append(q)
    return np.stack(Q).T

W = np.array([[1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
Qm = gram_schmidt(W)
print("Colonnes d'origine :\n", W)
print("Base orthonormale Q :\n", Qm)
print("Q.T @ Q ≈ I ?", np.allclose(Qm.T @ Qm, np.eye(2), atol=1e-8))

# Comparaison avec la QR de numpy
Qq, Rq = np.linalg.qr(W)
print("\nQR numpy : Q=\n", Qq, " (signes peuvent différer, mêmes directions)")

Colonnes d'origine :
 [[1. 1.]
 [0. 1.]
 [1. 0.]]
Base orthonormale Q :
 [[ 0.707  0.408]
 [ 0.     0.816]
 [ 0.707 -0.408]]
Q.T @ Q ≈ I ? True

QR numpy : Q=
 [[-0.707 -0.408]
 [-0.    -0.816]
 [-0.707  0.408]]  (signes peuvent différer, mêmes directions)


### Exercice corrigé

In [11]:
# ---- Exercice P5 : Gram-Schmidt sur [[1,1],[1,0]] ----
Vs = np.array([[1.0, 1.0], [1.0, 0.0]])
Qs = gram_schmidt(Vs)
print("P5-ex : Q =\n", Qs)
assert np.allclose(Qs.T @ Qs, np.eye(2), atol=1e-8)


P5-ex : Q =
 [[ 0.707  0.707]
 [ 0.707 -0.707]]


## 6 · Application : similarité cosinus (textes, embeddings)

Le **cosinus de similarité** mesure l'angle entre deux vecteurs, **indépendant
de leur longueur**. C'est la métrique reine pour :
- la recherche sémantique
- les embeddings (word2vec, BERT, etc.)
- le clustering de documents
- la détection de plagiat

`cos = 1` : mêmes directions (max similarité) ; `cos = 0` : perpendiculaires ;
`cos = -1` : opposés.

In [12]:
def cosine_sim(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# fr/mots simples : TF (term frequency)
doc1 = np.array([3, 1, 2, 0])   # [chat, chien, lapin, oiseau]
doc2 = np.array([2, 1, 3, 0])
doc3 = np.array([0, 0, 5, 4])
print(f"sim(doc1, doc2) = {cosine_sim(doc1, doc2):.3f}")
print(f"sim(doc1, doc3) = {cosine_sim(doc1, doc3):.3f}")
assert cosine_sim(doc1, doc2) > cosine_sim(doc1, doc3)

# Invariance à l'échelle : multiplier doc1 par 100 ne change pas le cosinus
print(f"sim(100*doc1, doc2) = {cosine_sim(100*doc1, doc2):.3f} (invariant !)")

sim(doc1, doc2) = 0.929
sim(doc1, doc3) = 0.417
sim(100*doc1, doc2) = 0.929 (invariant !)


### Exercice corrigé

In [13]:
# ---- Exercice P6 : similarite cosinus sur vecteurs-textes ----
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
w1 = np.array([2., 1, 0]); w2 = np.array([4., 2, 0]); w3 = np.array([0., 0, 1])
print(f"P6-ex : cos(w1,w2) = {cos(w1,w2):.4f} (proportionnels -> memes mots)")
print(f"       cos(w1,w3) = {cos(w1,w3):.4f} (orthogonal -> mots disjoints)")
assert np.isclose(cos(w1, w2), 1.0) and np.isclose(cos(w1, w3), 0.0)


P6-ex : cos(w1,w2) = 1.0000 (proportionnels -> memes mots)
       cos(w1,w3) = 0.0000 (orthogonal -> mots disjoints)


## 7 · Synthèse globale — l'essentiel & exemple intégrateur

### Ce qu'il faut retenir (les 6 points)

| Point | L'essentiel |
|---|---|
| **1 · Normes** | ‖x‖₂ = √Σxᵢ² ; normaliser = projeter sur la sphère unité |
| **2 · Produit scalaire** | u·v = ‖u‖‖v‖cos θ — le pont entre algèbre et géométrie |
| **3 · Orthogonalité & projection** | u ⊥ v ⇔ u·v = 0 ; projᵥ(u) = (u·v / v·v)·v |
| **4 · Distances** | euclidienne, Manhattan, Minkowski — la « bonne » dépend des données |
| **5 · Gram-Schmidt** | une famille libre → base orthonormée (utile en QR, PCA) |
| **6 · Similarité cosinus** | texte & embeddings : on compare des angles, pas des normes |

**Réflexe examen :** cosinus insensible à l'échelle, distance sensible.
L'exemple ci-dessous les met en compétition sur une recherche documentaire.

### Exercice corrigé

In [14]:
# ---- Exercice P7 : similarite cosinus invariante a l'echelle ----
u = np.array([1., 2, 3]); v = np.array([1., 1, 1]); w = np.array([10., 20, 30])
print(f"P7-ex : cos(u,w) = {u@w/(np.linalg.norm(u)*np.linalg.norm(w)):.4f} (u et w memes directions)")
print(f"       mais ||u-v|| = {np.linalg.norm(u-v):.3f} < ||w-v|| = {np.linalg.norm(w-v):.3f}")
assert np.isclose(u @ w / (np.linalg.norm(u)*np.linalg.norm(w)), 1.0)
assert np.linalg.norm(w-v) > np.linalg.norm(u-v)


P7-ex : cos(u,w) = 1.0000 (u et w memes directions)
       mais ||u-v|| = 2.236 < ||w-v|| = 35.819


### Exemple intégrateur (tous les points)

In [15]:
# ---- Exemple intégrateur : recherche documentaire (P1→P6) ----
docs = np.array([[1., 0, 1, 1],      # doc A
                 [2., 0, 2, 2],      # doc B = 2A  (même direction)
                 [0., 1, 0, 0],      # doc C
                 [1., 1, 1, 1]])     # doc D
q = np.array([1., 0, 1, 0])           # requête

norms = np.linalg.norm(docs, axis=1)                      # P1
cos = docs @ q / (norms * np.linalg.norm(q))              # P2 + P6 : similarité cosinus
ordre = np.argsort(-cos)
print(f"cosinus requête-docs : {np.round(cos, 3)} -> classement {ordre}")
assert np.isclose(cos[0], cos[1])                         # A et B : même direction
assert np.isclose(cos[2], 0.0)                            # C orthogonale à la requête (P3)
assert ordre[0] in (0, 1) and ordre[-1] == 2

projA = (q @ docs[0]) / (docs[0] @ docs[0]) * docs[0]     # P3 : projection
assert np.allclose(projA, (2/3) * docs[0])                # q·A = 2, ||A||² = 3
print(f"projection de q sur A = {projA} (= 2/3 · A)")

e1 = docs[0] / norms[0]                                   # P5 : Gram-Schmidt
v2 = docs[2] - (docs[2] @ e1) * e1
e2 = v2 / np.linalg.norm(v2)
assert abs(e1 @ e2) < 1e-12 and np.isclose(np.linalg.norm(e2), 1.0)
dist = np.linalg.norm(docs[0] - docs[2])                  # P4 : la distance, elle, distingue A et B
print(f"Gram-Schmidt OK ; ||A − C|| = {dist:.3f} alors que cos(A,C) = 0")
print("\n✅ T02 terminé — 6 points + synthèse globale.")

cosinus requête-docs : [0.816 0.816 0.    0.707] -> classement [0 1 3 2]
projection de q sur A = [0.667 0.    0.667 0.667] (= 2/3 · A)
Gram-Schmidt OK ; ||A − C|| = 2.000 alors que cos(A,C) = 0

✅ T02 terminé — 6 points + synthèse globale.
